# Macaron food photo model (v10)

Trains the photo model from the prepared photo packs made by **Macaron food photos** (no photo collecting here,
so the run starts training within minutes), and starts from **version 9's** learned weights instead of from scratch.

- Photos come in three groups fixed once: **train** (learn), **check** (pick the best version), **final** (scored once at the end).
- At the end it writes `report.json`: the final score, the score of each dish, the dishes it mixes up most, and how often
  it is right at each confidence level, so the app can say "not sure" honestly.
- Saved for the app: `food.onnx`, `labels.json`, `parts.json`. Also `weights.pt`, so the next version can start from this one.

In [ ]:
!pip -q install -U timm onnx onnxruntime


In [ ]:
import json, os, io, csv, time, random, tarfile, numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image

SEED = 7; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
T0 = time.time(); hours = lambda: (time.time() - T0) / 3600
BUDGET_H = 11.0
DRY = bool(os.environ.get("MACARON_DRY"))
if DEVICE != "cuda" and not DRY:
    raise SystemExit("STOP: no GPU. Set Accelerator to GPU T4 x2 and Run All again. Nothing was used up.")

ARCH = "mobilenetv4_conv_medium.e500_r256_in1k"
BATCH = 64
EPOCHS_HEAD, EPOCHS_ALL = 1, 12
PARTS_WEIGHT = 0.5
MIN_TRAIN = 20                  # a dish needs at least this many training photos to be learned

def find(name):
    """Kaggle puts attached inputs under /kaggle/input/<slug>/, sometimes one folder deeper."""
    for root, _, files in os.walk("/kaggle/input"):
        if name in files: return os.path.join(root, name)
    return None
PHOTOS = os.environ.get("MACARON_PHOTOS") or (os.path.dirname(find("manifest.csv") or "") or None)
V9_WEIGHTS = os.environ.get("MACARON_V9") or find("weights_so_far.pt") or find("weights.pt")
V9_LABELS = os.environ.get("MACARON_V9_LABELS") or (os.path.join(os.path.dirname(V9_WEIGHTS), "labels.json") if V9_WEIGHTS else None)
if not PHOTOS: raise SystemExit("STOP: the prepared photos are not attached. Add the 'Macaron food photos' notebook output as input.")
print("photos:", PHOTOS, "| starting weights:", V9_WEIGHTS or "none (learning from scratch)")


## Read the prepared photos

In [ ]:
# ---- read the packs ----
meta = json.load(open(os.path.join(PHOTOS, "meta.json")))
PART_NAMES = meta["part_names"]
rows = list(csv.DictReader(open(os.path.join(PHOTOS, "manifest.csv"))))
LOCAL = "/tmp/photos"; os.makedirs(LOCAL, exist_ok=True)
for p in sorted({r["pack"] for r in rows}):
    with tarfile.open(os.path.join(PHOTOS, p)) as t: t.extractall(os.path.join(LOCAL, p[:-4]))
print(f"unpacked {len(rows)} photos in {hours() * 60:.0f} minutes")

cnt = meta["labels"]
CLASSES = sorted(c for c, n in cnt.items() if not c.startswith("parts:") and n["train"] >= MIN_TRAIN and n["final"] >= 4)
PO_LABS = sorted(c for c, n in cnt.items() if c.startswith("parts:") and n["train"] >= MIN_TRAIN)
IDX = {c: i for i, c in enumerate(CLASSES + PO_LABS)}; NC = len(CLASSES)
PI = {p: i for i, p in enumerate(PART_NAMES)}
def pv(tags):
    v = torch.zeros(len(PART_NAMES))
    for t in tags: v[PI[t]] = 1
    return v
PT = torch.stack([pv(meta["dish_parts"].get(c, [])) for c in CLASSES] + [pv(meta["parts_only"][c]) for c in PO_LABS])
left_out = sorted(c for c in cnt if not c.startswith("parts:") and c not in IDX)
print(NC, "dishes to learn,", len(PO_LABS), "parts-only sets.", len(left_out), "dishes left out for too few photos:", ", ".join(left_out[:60]))

def group(g): return [(os.path.join(LOCAL, r["pack"][:-4], r["file"]), IDX[r["label"]]) for r in rows if r["group"] == g and r["label"] in IDX]
train_rows, check_rows, final_rows = group("train"), group("check"), group("final")
print(len(train_rows), "train,", len(check_rows), "check,", len(final_rows), "final photos")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
def square(img):
    """Centre square then 224, exactly how the app prepares a photo."""
    w, h = img.size; m = min(w, h)
    return img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2)).resize((224, 224), Image.BILINEAR)
train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.4, 1.0)), T.RandomHorizontalFlip(), T.TrivialAugmentWide(),
                      T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.25)])
test_tf = T.Compose([T.Lambda(square), T.ToTensor(), T.Normalize(MEAN, STD)])
class Files(Dataset):
    def __init__(self, rows, tf): self.rows, self.tf = rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        fn, y = self.rows[k]; return self.tf(Image.open(fn).convert("RGB")), y
NW = 0 if DRY else 4
train_dl = DataLoader(Files(train_rows, train_tf), BATCH, shuffle=True, num_workers=NW, drop_last=True, pin_memory=not DRY)
check_dl = DataLoader(Files(check_rows, test_tf), BATCH, num_workers=NW)
final_dl = DataLoader(Files(final_rows, test_tf), BATCH, num_workers=NW)


## Train, starting from version 9

In [ ]:
# ---- the network, started from version 9 ----
class TwoAnswers(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.body = timm.create_model(ARCH, pretrained=pretrained, num_classes=0)
        self.body.eval()
        with torch.no_grad(): n = self.body(torch.zeros(2, 3, 224, 224)).shape[1]
        self.body.train()
        self.dish = nn.Linear(n, NC); self.parts = nn.Linear(n, len(PART_NAMES))
    def forward(self, x):
        f = self.body(x); return self.dish(f), self.parts(f)

model = TwoAnswers(pretrained=V9_WEIGHTS is None)
reused = 0
if V9_WEIGHTS:
    old = torch.load(V9_WEIGHTS, map_location="cpu")
    body = {k[5:]: v for k, v in old.items() if k.startswith("body.")}
    model.body.load_state_dict(body)
    old_names = [x["food"] for x in json.load(open(V9_LABELS))] if V9_LABELS and os.path.exists(V9_LABELS) else []
    with torch.no_grad():
        if old["parts.weight"].shape == model.parts.weight.shape:
            model.parts.weight.copy_(old["parts.weight"]); model.parts.bias.copy_(old["parts.bias"])
        pos = {c: i for i, c in enumerate(old_names)}
        for j, c in enumerate(CLASSES):           # a dish version 9 knew keeps what it learned; new dishes start fresh
            if c in pos and pos[c] < old["dish.weight"].shape[0]:
                model.dish.weight[j].copy_(old["dish.weight"][pos[c]]); model.dish.bias[j].copy_(old["dish.bias"][pos[c]]); reused += 1
    print(f"started from version 9: {reused} of {NC} dishes already known, {NC - reused} new")
model = model.to(DEVICE)
HEAD = {n for n, _ in model.named_parameters() if n.startswith(("dish.", "parts."))}
ema = timm.utils.ModelEmaV3(model, decay=0.999)
dish_loss = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)
parts_loss = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
PTD = PT.to(DEVICE)

def losses(out, y):
    d, p = out; yd = torch.where(y < NC, y, torch.full_like(y, -100))
    l = PARTS_WEIGHT * parts_loss(p.float(), PTD[y])
    if (yd >= 0).any(): l = l + dish_loss(d.float(), yd)
    return l

def scores(net, dl):
    """All dish scores for a group, as probabilities, with the true dish."""
    net.eval(); P, Y = [], []
    with torch.no_grad():
        for x, y in dl:
            m = y < NC
            if not m.any(): continue
            d, _ = net(x[m].to(DEVICE)); P.append(torch.softmax(d.float(), 1).cpu()); Y.append(y[m])
    return torch.cat(P), torch.cat(Y)
def top(P, Y, k): return (P.topk(k, 1).indices == Y[:, None]).any(1).float().mean().item()

BEST = {"top1": -1}
def run(epochs, lr, head_only):
    for n, p in model.named_parameters(): p.requires_grad = (not head_only) or n in HEAD
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.02)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(1, epochs * len(train_dl)))
    for ep in range(epochs):
        t_ep = time.time(); model.train(); total = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"): out = model(x)
            loss = losses(out, y); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update(model); total += loss.item()
        P, Y = scores(ema.module, check_dl); a1, a3 = top(P, Y, 1), top(P, Y, 3)
        print(f"epoch {ep + 1}/{epochs}  loss {total / max(1, len(train_dl)):.3f}  check photos: right first time {a1:.1%}  in top 3 {a3:.1%}  ({hours():.1f} h so far)")
        if a1 > BEST["top1"]:
            BEST.update(top1=a1, top3=a3); torch.save(ema.module.state_dict(), "/kaggle/working/weights.pt" if os.path.isdir("/kaggle/working") else "/tmp/weights.pt")
        if hours() + 1.3 * (time.time() - t_ep) / 3600 + 0.4 > BUDGET_H and ep + 1 < epochs:
            print("Stopping early to finish inside Kaggle's 12 hours."); break

run(EPOCHS_HEAD, 2e-3, head_only=True)
run(EPOCHS_ALL, 5e-4 if V9_WEIGHTS else 6e-4, head_only=False)


## Final score

In [ ]:
# ---- the final score, once, on photos never used before ----
W = "/kaggle/working/weights.pt" if os.path.isdir("/kaggle/working") else "/tmp/weights.pt"
best = TwoAnswers(pretrained=False); best.load_state_dict(torch.load(W, map_location="cpu")); best = best.to(DEVICE).eval()
P, Y = scores(best, final_dl); pred = P.argmax(1); conf = P.max(1).values
f1, f3 = top(P, Y, 1), top(P, Y, 3)
per = {}
for c in range(NC):
    m = Y == c
    if m.any(): per[CLASSES[c]] = {"photos": int(m.sum()), "right": round((pred[m] == c).float().mean().item(), 3)}
mix = {}
for a, b in zip(Y.tolist(), pred.tolist()):
    if a != b: mix[(CLASSES[a], CLASSES[b])] = mix.get((CLASSES[a], CLASSES[b]), 0) + 1
bins = []
for lo in [0, .2, .4, .6, .8, .9]:
    hi = {0: .2, .2: .4, .4: .6, .6: .8, .8: .9, .9: 1.01}[lo]; m = (conf >= lo) & (conf < hi)
    if m.any(): bins.append({"confidence": f"{lo:.0%} to {min(hi, 1):.0%}", "photos": int(m.sum()), "right": round((pred[m] == Y[m]).float().mean().item(), 3)})
report = {"version": 10, "final_photos": int(len(Y)), "right_first_time": round(f1, 4), "in_top_3": round(f3, 4),
          "best_check": BEST, "dishes": NC, "started_from_v9": bool(V9_WEIGHTS), "dishes_known_before": reused,
          "weakest": sorted(per.items(), key=lambda kv: kv[1]["right"])[:60],
          "mixed_up_most": [{"dish": a, "taken_for": b, "times": n} for (a, b), n in sorted(mix.items(), key=lambda kv: -kv[1])[:60]],
          "confidence": bins, "left_out_for_too_few_photos": left_out}
# fair comparison: version 9 and version 10 on the same never-seen photos of dishes both know
if V9_WEIGHTS and old_names:
    old_m = TwoAnswers(pretrained=False); old_m.dish = nn.Linear(old_m.dish.in_features, old["dish.weight"].shape[0]); old_m.load_state_dict(old); old_m = old_m.to(DEVICE).eval()
    Po, Yo = scores(old_m, final_dl)
    both = torch.tensor([CLASSES[int(y)] in pos for y in Yo])
    if both.any():
        o_pred = [old_names[int(i)] for i in Po[both].argmax(1)]; o_top3 = [[old_names[int(i)] for i in r] for r in Po[both].topk(3, 1).indices]
        truth = [CLASSES[int(y)] for y in Yo[both]]
        v9_1 = np.mean([a == b for a, b in zip(o_pred, truth)]); v9_3 = np.mean([t in r for t, r in zip(truth, o_top3)])
        v10_1, v10_3 = top(P[both], Y[both], 1), top(P[both], Y[both], 3)
        report["same_photos_v9_vs_v10"] = {"photos": int(both.sum()), "v9_first": round(float(v9_1), 4), "v9_top3": round(float(v9_3), 4), "v10_first": round(v10_1, 4), "v10_top3": round(v10_3, 4)}
        print(f"SAME PHOTOS ({int(both.sum())}): version 9 right first time {v9_1:.1%}, top 3 {v9_3:.1%}  |  version 10 {v10_1:.1%}, top 3 {v10_3:.1%}")
json.dump(report, open("report.json", "w"), indent=1, ensure_ascii=False)
print(f"\nFINAL SCORE on {len(Y)} never-seen photos: right first time {f1:.1%}, in top 3 {f3:.1%}")
for b in bins: print(f"  when {b['confidence']} sure: right {b['right']:.0%} of {b['photos']} photos")
print("weakest dishes:"); [print(f"  {v['right']:5.0%}  {k}") for k, v in report["weakest"][:25]]
print("mixed up most:"); [print(f"  {x['dish']} -> {x['taken_for']} ({x['times']})") for x in report["mixed_up_most"][:20]]


## Save for the app

In [ ]:
# ---- save for the app ----
import onnx, onnxruntime as ort
best.eval().cpu()
class ForApp(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): d, p = self.m(x); return d, torch.sigmoid(p)
app_model = ForApp(best).eval()
kw = dict(input_names=["input"], output_names=["logits", "parts"], opset_version=17)
try: torch.onnx.export(app_model, torch.zeros(1, 3, 224, 224), "food.onnx", dynamo=False, **kw)
except TypeError: torch.onnx.export(app_model, torch.zeros(1, 3, 224, 224), "food.onnx", **kw)
onnx.checker.check_model(onnx.load("food.onnx"))
import re
json.dump([{"label": re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_"), "food": c} for c in CLASSES], open("labels.json", "w"), indent=1)
json.dump({"parts": PART_NAMES}, open("parts.json", "w"), indent=1)
x, _ = next(iter(final_dl)); a, _ = ort.InferenceSession("food.onnx").run(None, {"input": x[:1].numpy()})
b, _ = app_model(x[:1]); print("largest difference between saved and trained model:", float(np.abs(a - b.detach().numpy()).max()))
print(f"food.onnx: {os.path.getsize('food.onnx') / 1e6:.1f} MB.  labels.json: {NC} classes.  Took {hours():.1f} h in total.")
